In [1]:
import sparkocr
import pyspark
import sys
import sparknlp_jsl
from pyspark.sql import SparkSession
from sparkocr import start
import os
import base64
from sparkocr.transformers import *
from pyspark.ml import PipelineModel
from pyspark.sql import functions as F
from sparkocr.enums import *
from sparkocr.utils import display_images

from sparknlp.annotator import *
from sparknlp_jsl.annotator import *
from sparknlp.base import *

In [2]:
spark = sparkocr.start(
    secret=os.environ.get('SPARK_OCR_SECRET', ''),
    nlp_version=os.environ.get('PUBLIC_VERSION', ''),
    nlp_secret=os.environ.get('SECRET', ''),
    nlp_internal=os.environ.get('JSL_VERSION', '')
)
spark

Spark version: 3.1.1
Spark NLP version: 5.5.3
Spark NLP for Healthcare version: 6.0.2
Spark OCR version: 6.0.0

:: loading settings :: url = jar:file:/usr/local/lib/python3.8/dist-packages/pyspark/jars/ivy-2.4.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0507a1f6-e883-4eb9-9bb5-404915e759d4;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp_2.12;6.0.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central
	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.g

	[SUCCESSFUL ] com.johnsnowlabs.nlp#tensorflow-cpu_2.12;0.4.4!tensorflow-cpu_2.12.jar (6887ms)
downloading https://repo1.maven.org/maven2/com/microsoft/onnxruntime/onnxruntime/1.19.2/onnxruntime-1.19.2.jar ...
	[SUCCESSFUL ] com.microsoft.onnxruntime#onnxruntime;1.19.2!onnxruntime.jar (2160ms)
downloading https://repo1.maven.org/maven2/com/johnsnowlabs/nlp/jsl-llamacpp-cpu_2.12/0.1.6/jsl-llamacpp-cpu_2.12-0.1.6.jar ...
	[SUCCESSFUL ] com.johnsnowlabs.nlp#jsl-llamacpp-cpu_2.12;0.1.6!jsl-llamacpp-cpu_2.12.jar (42ms)
downloading https://repo1.maven.org/maven2/com/johnsnowlabs/nlp/jsl-openvino-cpu_2.12/0.1.0/jsl-openvino-cpu_2.12-0.1.0.jar ...
	[SUCCESSFUL ] com.johnsnowlabs.nlp#jsl-openvino-cpu_2.12;0.1.0!jsl-openvino-cpu_2.12.jar (1292ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-kms/1.12.500/aws-java-sdk-kms-1.12.500.jar ...
	[SUCCESSFUL ] com.amazonaws#aws-java-sdk-kms;1.12.500!aws-java-sdk-kms.jar (27ms)
downloading https://repo1.maven.org/maven2/com/amazon

downloading https://repo1.maven.org/maven2/io/grpc/grpc-core/1.53.0/grpc-core-1.53.0.jar ...
	[SUCCESSFUL ] io.grpc#grpc-core;1.53.0!grpc-core.jar (61ms)
downloading https://repo1.maven.org/maven2/com/google/api/gax/2.23.2/gax-2.23.2.jar ...
	[SUCCESSFUL ] com.google.api#gax;2.23.2!gax.jar (21ms)
downloading https://repo1.maven.org/maven2/com/google/api/gax-grpc/2.23.2/gax-grpc-2.23.2.jar ...
	[SUCCESSFUL ] com.google.api#gax-grpc;2.23.2!gax-grpc.jar (11ms)
downloading https://repo1.maven.org/maven2/com/google/auth/google-auth-library-credentials/1.16.0/google-auth-library-credentials-1.16.0.jar ...
	[SUCCESSFUL ] com.google.auth#google-auth-library-credentials;1.16.0!google-auth-library-credentials.jar (8ms)
downloading https://repo1.maven.org/maven2/com/google/auth/google-auth-library-oauth2-http/1.16.0/google-auth-library-oauth2-http-1.16.0.jar ...
	[SUCCESSFUL ] com.google.auth#google-auth-library-oauth2-http;1.16.0!google-auth-library-oauth2-http.jar (13ms)
downloading https://rep

	98 artifacts copied, 0 already retrieved (624481kB/731ms)
25/06/24 17:50:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
def deidentification_nlp_pipeline(input_column, prefix = ""):
    document_assembler = DocumentAssembler() \
        .setInputCol(input_column) \
        .setOutputCol(prefix + "document")

    # Sentence Detector annotator, processes various sentences per line
    sentence_detector = SentenceDetector() \
        .setInputCols([prefix + "document"]) \
        .setOutputCol(prefix + "sentence")

    tokenizer = Tokenizer() \
        .setInputCols([prefix + "sentence"]) \
        .setOutputCol(prefix + "token")

    # Clinical word embeddings
    word_embeddings = WordEmbeddingsModel.pretrained("embeddings_clinical", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token"]) \
        .setOutputCol(prefix + "embeddings")
    # NER model trained on i2b2 (sampled from MIMIC) dataset
    clinical_ner = MedicalNerModel.pretrained("ner_deid_large", "en", "clinical/models") \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "embeddings"]) \
        .setOutputCol(prefix + "ner")

    custom_ner_converter = NerConverterInternal() \
        .setInputCols([prefix + "sentence", prefix + "token", prefix + "ner"]) \
        .setOutputCol(prefix + "ner_chunk") \
        .setWhiteList(['NAME', 'AGE', 'CONTACT', 'LOCATION', 'PROFESSION', 'PERSON', 'DATE'])

    nlp_pipeline = Pipeline(stages=[
            document_assembler,
            sentence_detector,
            tokenizer,
            word_embeddings,
            clinical_ner,
            custom_ner_converter
        ])
    empty_data = spark.createDataFrame([[""]]).toDF(input_column)
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

In [4]:
# Read dicom as image
dicom_to_image = DicomToImage() \
    .setInputCol("content") \
    .setOutputCol("image_raw") \
    .setMetadataCol("metadata") \
    .setDeIdentifyMetadata(True)

adaptive_thresholding = ImageAdaptiveThresholding() \
    .setInputCol("image_raw") \
    .setOutputCol("corrected_image") \
    .setBlockSize(47) \
    .setOffset(4) \
    .setKeepInput(True)

# Extract text from image
ocr = ImageToText() \
    .setInputCol("corrected_image") \
    .setOutputCol("text")

# Found coordinates of sensitive data
position_finder = PositionFinder() \
    .setInputCols("ner_chunk") \
    .setOutputCol("coordinates") \
    .setPageMatrixCol("positions")

# Found sensitive data using DeIdentificationModel
deidentification_rules = DeIdentificationModel.pretrained("deidentify_rb_no_regex", "en", "clinical/models") \
    .setInputCols(["metadata_sentence", "metadata_token", "metadata_ner_chunk"]) \
    .setOutputCol("deidentified_metadata_raw")

finisher = Finisher() \
    .setInputCols(["deidentified_metadata_raw"]) \
    .setOutputCols("deidentified_metadata") \
    .setOutputAsArray(False) \
    .setValueSplitSymbol("") \
    .setAnnotationSplitSymbol("")

# Draw filled rectangle for hide sensitive data
drawRegions = ImageDrawRegions()  \
    .setInputCol("image_raw")  \
    .setInputRegionsCol("coordinates")  \
    .setOutputCol("image_with_regions")  \
    .setFilledRect(True) \
    .setRectColor(Color.black)

# Store image back to Dicom document
imageToDicom = ImageToDicom() \
    .setInputCol("image_with_regions") \
    .setOutputCol("dicom")

# OCR pipeline
deid_pipeline = PipelineModel(stages=[
    dicom_to_image,
    adaptive_thresholding,
    ocr,
    deidentification_nlp_pipeline(input_column="text"),
    position_finder,
    drawRegions,
    #imageToDicom  # Commented for able to demonstrate intermidiate results before aggregation
])

deidentify_rb_no_regex download started this may take some time.
[ | ]

25/06/24 17:55:30 WARN ApacheUtils: NoSuchMethodException was thrown when disabling normalizeUri. This indicates you are using an old version (< 4.5.8) of Apache http client. It is recommended to use http client version >= 4.5.9 to avoid the breaking change introduced in apache client 4.5.7 and the latency in exception handling. See https://github.com/aws/aws-sdk-java/issues/1919 for more information


[ / ]deidentify_rb_no_regex download started this may take some time.
Approximate size to download 8.9 KB
Download done! Loading the resource.
[OK!]
embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ | ]

25/06/24 17:55:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 17:55:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 17:55:35 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB
[ \ ]Download done! Loading the resource.
[OK!]
ner_deid_large download started this may take some time.
[ | ]

25/06/24 17:57:40 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/06/24 17:57:40 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


ner_deid_large download started this may take some time.
Approximate size to download 14.1 MB
[ / ]Download done! Loading the resource.
[ — ]

2025-06-24 17:57:46.673804: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-24 17:57:46.776923: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_nam

[OK!]
